# **0. 환경설정**

In [ ]:
# [CELL 1] 라이브러리 설치 & 임포트
# ====================================================================

# Plotly 설치
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "plotly", "kaleido"])

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import numpy as np
import io

print("✅ 라이브러리 설치 완료!")

✅ 라이브러리 설치 완료!


In [ ]:
# [CELL 2] Google Drive 연동 (선택사항)
# ====================================================================

# Google Drive에서 파일을 가져올 경우 아래 코드 실행
# Google Drive 연동을 원하면 주석 풀기

from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive 연동 완료!")

# 파일 경로 (Google Drive에 저장된 경우)
file_path = '/content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/고객 세그먼트 분류/김민주/[공동작업] KCGS_평가등급_2022-2025_채점결과.xlsx'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive 연동 완료!


In [ ]:
# [CELL 3] 파일 로드 & 데이터 준비
# ====================================================================

# Colab 업로드 파일 사용 (위의 파일 업로드 위젯 사용)
# 또는 아래와 같이 로컬 경로 지정

import os

# Colab 임시 폴더에서 파일 찾기
file_path = '/content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/고객 세그먼트 분류/김민주/[공동작업] KCGS_평가등급_2022-2025_채점결과.xlsx'

# 파일이 없으면 업로드 안내
if not os.path.exists(file_path):
    print("⚠️ 파일을 찾을 수 없습니다. 아래 셀에서 파일을 업로드하세요:")
    print("from google.colab import files")
    print("files.upload()")
else:
    print(f"✅ 파일 발견: {file_path}")

✅ 파일 발견: /content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/고객 세그먼트 분류/김민주/[공동작업] KCGS_평가등급_2022-2025_채점결과.xlsx


In [ ]:
# ====================================================================

df_raw = pd.read_excel(file_path, sheet_name='260907_고객 세그먼트 분류용')

print(f"📊 데이터 크기: {df_raw.shape}")
print(f"📋 컬럼: {df_raw.columns.tolist()[:10]}...")

# 정제
df_raw['기업명'] = df_raw['기업명'].astype(str).str.strip()
df_raw['산업코드'] = df_raw['산업코드'].astype(str).str.strip()

# 업종 분류 (C261 = 반도체)
df_raw['업종분류'] = df_raw['산업코드'].apply(
    lambda x: '반도체' if str(x).startswith('C261') else '비반도체'
)

print("✅ 데이터 정제 완료!")
print(df_raw.head())

📊 데이터 크기: (352, 22)
📋 컬럼: ['기업명', '기업코드', '평가년도', 'ESG등급', '환경', '사회', '지배구조', 'ESG 등급조정', '환경 등급조정', '사회 등급조정']...
✅ 데이터 정제 완료!
     기업명    기업코드    평가년도 ESG등급  환경  사회 지배구조  ESG 등급조정  환경 등급조정  사회 등급조정  ...  \
0   가온전선   500.0  2022.0     B   C  B+   B+       NaN      NaN      NaN  ...   
1   가온전선   500.0  2023.0    B+  B+   A    B       NaN      NaN      NaN  ...   
2   가온전선   500.0  2024.0     B   A   A    C       NaN      NaN      NaN  ...   
3   가온전선   500.0  2025.0    B+   A  A+    B       NaN      NaN      NaN  ...   
4  경동나비엔  9450.0  2022.0     C   C   B    D       NaN      NaN      NaN  ...   

        종가      대비   등락률          시가총액  지속가능경영보고서_발행  기업지배구조보고서_발행  보고서_발행_유형  \
0      NaN     NaN   NaN           NaN           1.0           0.0     지속가능경영   
1      NaN     NaN   NaN           NaN           1.0           0.0     지속가능경영   
2      NaN     NaN   NaN           NaN           1.0           1.0     둘 다 발행   
3  82400.0 -2600.0 -3.06  1.363150e+12           1.0           1.0

In [ ]:
# [CELL 6] 한글 폰트 설정 (Colab 환경)
# ====================================================================

import matplotlib.pyplot as plt

# Colab에서 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'

# Plotly 한글 폰트 설정
pio.templates.default = "plotly_white"

print("✅ 폰트 설정 완료!")

✅ 폰트 설정 완료!


In [ ]:
# [CELL 7] EDA 1: 업종 분류
# ====================================================================

print("="*70)
print("[EDA 1] 반도체 vs 비반도체 업종 분류")
print("="*70)

df_2025 = df_raw[df_raw['평가년도'] == 2025].copy()
industry_summary = df_2025['업종분류'].value_counts().reset_index()
industry_summary.columns = ['업종분류', '기업수']
industry_summary['비중(%)'] = (
    industry_summary['기업수'] / industry_summary['기업수'].sum() * 100
).round(1)

print(f"\n2025년 기준 업종별 기업 수:")
print(industry_summary.to_string(index=False))
print(f"\n총 기업 수: {industry_summary['기업수'].sum()}개사")

[EDA 1] 반도체 vs 비반도체 업종 분류

2025년 기준 업종별 기업 수:
업종분류  기업수  비중(%)
비반도체   74   83.1
 반도체   15   16.9

총 기업 수: 89개사


In [ ]:
# [CELL 8] EDA 2: 비반도체 2024년 보충
# ====================================================================

print("\n" + "="*70)
print("[EDA 2] 비반도체 업종: 2025년 데이터 vs 2024년 데이터 보충")
print("="*70)

df_non_semi = df_raw[df_raw['업종분류'] == '비반도체'].copy()
non_semi_2025 = set(df_non_semi[df_non_semi['평가년도'] == 2025]['기업명'].unique())
print(f"\n2025년 데이터 보유 비반도체 기업: {len(non_semi_2025)}개사")

# 최신 데이터 추출 (2025 없으면 2024)
df_non_semi_latest = []
for company in df_non_semi['기업명'].unique():
    company_data = df_non_semi[df_non_semi['기업명'] == company].sort_values('평가년도', ascending=False)
    if len(company_data) > 0:
        df_non_semi_latest.append(company_data.iloc[0])

df_non_semi_latest = pd.DataFrame(df_non_semi_latest)
df_non_semi_2024_backfill = df_non_semi_latest[df_non_semi_latest['평가년도'] == 2024]
print(f"2024년으로 보충된 비반도체 기업: {len(df_non_semi_2024_backfill)}개사")
print(f"\n보충된 기업:")
print(df_non_semi_2024_backfill[['기업명', '평가년도', 'ESG등급']].to_string(index=False))


[EDA 2] 비반도체 업종: 2025년 데이터 vs 2024년 데이터 보충

2025년 데이터 보유 비반도체 기업: 74개사
2024년으로 보충된 비반도체 기업: 1개사

보충된 기업:
 기업명   평가년도 ESG등급
KH바텍 2024.0     A


In [ ]:
# [CELL 9] EDA 3: ESG 등급 분포
# ====================================================================

print("\n" + "="*70)
print("[EDA 3] ESG 등급 분포: 반도체 vs 비반도체")
print("="*70)

df_semi = df_raw[df_raw['업종분류'] == '반도체'].copy()
df_semi_latest = []
for company in df_semi['기업명'].unique():
    company_data = df_semi[df_semi['기업명'] == company].sort_values('평가년도', ascending=False)
    if len(company_data) > 0:
        df_semi_latest.append(company_data.iloc[0])
df_semi_latest = pd.DataFrame(df_semi_latest)

grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']

def classify_grades(df, grade_order):
    df = df.copy()
    df['ESG등급'] = pd.Categorical(df['ESG등급'], categories=grade_order, ordered=True)
    grade_dist = df['ESG등급'].value_counts().reindex(grade_order).fillna(0).reset_index()
    grade_dist.columns = ['ESG등급', '기업수']
    grade_dist['기업수'] = grade_dist['기업수'].astype(int)
    grade_dist['비중(%)'] = (grade_dist['기업수'] / grade_dist['기업수'].sum() * 100).round(1)
    return grade_dist

grade_semi = classify_grades(df_semi_latest, grade_order)
grade_non_semi = classify_grades(df_non_semi_latest, grade_order)

print(f"\n반도체 업종 ESG 등급 분포:")
print(grade_semi.to_string(index=False))
print(f"총: {grade_semi['기업수'].sum()}개사")

print(f"\n비반도체 업종 ESG 등급 분포:")
print(grade_non_semi.to_string(index=False))
print(f"총: {grade_non_semi['기업수'].sum()}개사")


[EDA 3] ESG 등급 분포: 반도체 vs 비반도체

반도체 업종 ESG 등급 분포:
ESG등급  기업수  비중(%)
   A+    0    0.0
    A    3   18.8
   B+    1    6.2
    B    1    6.2
    C    6   37.5
    D    5   31.2
총: 16개사

비반도체 업종 ESG 등급 분포:
ESG등급  기업수  비중(%)
   A+    1    1.3
    A   17   22.4
   B+   12   15.8
    B    6    7.9
    C   10   13.2
    D   30   39.5
총: 76개사


In [ ]:
# [CELL 10] EDA 4: E/S/G 세부 등급
# ====================================================================

print("\n" + "="*70)
print("[EDA 4] E/S/G 세부 등급 비교: 반도체 vs 비반도체")
print("="*70)

esg_categories = ['환경', '사회', '지배구조']

for category in esg_categories:
    print(f"\n【{category} 등급 분포】")

    df_s = df_semi_latest.copy()
    df_ns = df_non_semi_latest.copy()

    df_s[category] = pd.Categorical(df_s[category], categories=grade_order, ordered=True)
    esg_semi = df_s[category].value_counts().reindex(grade_order).fillna(0)

    df_ns[category] = pd.Categorical(df_ns[category], categories=grade_order, ordered=True)
    esg_non_semi = df_ns[category].value_counts().reindex(grade_order).fillna(0)

    esg_compare = pd.DataFrame({
        '등급': esg_semi.index,
        '반도체': esg_semi.values.astype(int),
        '비반도체': esg_non_semi.values.astype(int)
    })
    esg_compare['반도체_비중(%)'] = (esg_compare['반도체'] / esg_compare['반도체'].sum() * 100).round(1)
    esg_compare['비반도체_비중(%)'] = (esg_compare['비반도체'] / esg_compare['비반도체'].sum() * 100).round(1)

    print(esg_compare.to_string(index=False))


[EDA 4] E/S/G 세부 등급 비교: 반도체 vs 비반도체

【환경 등급 분포】
등급  반도체  비반도체  반도체_비중(%)  비반도체_비중(%)
A+    0     2        0.0         2.6
 A    3    17       18.8        22.4
B+    4    14       25.0        18.4
 B    1     8        6.2        10.5
 C    8    23       50.0        30.3
 D    0    12        0.0        15.8

【사회 등급 분포】
등급  반도체  비반도체  반도체_비중(%)  비반도체_비중(%)
A+    3    20       18.8        26.3
 A    4    13       25.0        17.1
B+    1     6        6.2         7.9
 B    2     5       12.5         6.6
 C    3    11       18.8        14.5
 D    3    21       18.8        27.6

【지배구조 등급 분포】
등급  반도체  비반도체  반도체_비중(%)  비반도체_비중(%)
A+    0     0        0.0         0.0
 A    1     7        6.2         9.2
B+    3    14       18.8        18.4
 B    1    11        6.2        14.5
 C    2     7       12.5         9.2
 D    9    37       56.2        48.7


In [ ]:
# [CELL 11] 통합 데이터 준비
# ====================================================================

df_semi_latest['업종분류'] = '반도체'
df_non_semi_latest['업종분류'] = '비반도체'
df_segment = pd.concat([df_semi_latest, df_non_semi_latest], ignore_index=True)
df_segment['ESG등급'] = pd.Categorical(df_segment['ESG등급'], categories=grade_order, ordered=True)

# 시가총액 수치화
def parse_market_cap(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if '조' in x:
        return float(x.replace('조', '').replace('원', '').strip()) * 1_000_000_000_000
    elif '억' in x:
        return float(x.replace('억', '').replace('원', '').strip()) * 100_000_000
    else:
        try:
            return float(x)
        except:
            return np.nan

df_segment['시가총액_수치'] = df_segment['시가총액'].apply(parse_market_cap)

print("✅ 통합 데이터 준비 완료!")

✅ 통합 데이터 준비 완료!


# **[STEP 1] 기초 EDA**
- 반도체 vs 비반도체 분석
- ESG 등급 분포
- 산업코드별 분석
- 시가총액별 분석
- Segment별 분석

In [ ]:
# [CELL 12] 시각화 1: 업종 분포
# ====================================================================

print("\n" + "="*70)
print("[시각화] 차트 생성 중...")
print("="*70)

color_map_industry = {'반도체': '#1F77B4', '비반도체': '#FF7F0E'}

fig1 = px.bar(
    industry_summary,
    x='업종분류',
    y='기업수',
    text='비중(%)',
    title='<b>[EDA 1] 반도체 vs 비반도체 업종 분포 (2025년 기준)</b>',
    labels={'기업수': '기업 수 (개사)', '업종분류': '업종분류'},
    color='업종분류',
    color_discrete_map=color_map_industry
)
fig1.update_traces(texttemplate='%{y}개사<br>(%{text:.1f}%)', textposition='outside')
fig1.update_layout(
    font=dict(family='Arial'),
    yaxis=dict(range=[0, industry_summary['기업수'].max() * 1.3]),
    showlegend=False,
    height=500,
    width=800
)
fig1.show()

print("✅ 차트 1 완료")


[시각화] 차트 생성 중...


✅ 차트 1 완료


In [ ]:
# [CELL 13] 시각화 2: ESG 등급 비교
# ====================================================================

grade_compare = pd.concat([
    grade_semi.assign(업종분류='반도체'),
    grade_non_semi.assign(업종분류='비반도체')
])

fig2 = px.bar(
    grade_compare,
    x='ESG등급',
    y='기업수',
    color='업종분류',
    barmode='group',
    title='<b>[EDA 3] ESG 등급 분포 비교: 반도체 vs 비반도체</b>',
    labels={'기업수': '기업 수 (개사)', 'ESG등급': 'ESG 통합 등급'},
    color_discrete_map=color_map_industry
)
fig2.update_layout(
    font=dict(family='Arial'),
    xaxis={'categoryorder': 'array', 'categoryarray': grade_order},
    height=500,
    width=800
)
fig2.show()

print("✅ 차트 2 완료")

✅ 차트 2 완료


In [ ]:
# [CELL 14] 시각화 3: E/S/G 세부 비교 (Heatmap)
# ====================================================================

esg_heatmap_semi = []
for category in ['환경', '사회', '지배구조']:
    df_s = df_semi_latest.copy()
    df_s[category] = pd.Categorical(df_s[category], categories=grade_order, ordered=True)
    dist = df_s[category].value_counts().reindex(grade_order).fillna(0).astype(int)
    esg_heatmap_semi.append(dist.values)

esg_heatmap_non_semi = []
for category in ['환경', '사회', '지배구조']:
    df_ns = df_non_semi_latest.copy()
    df_ns[category] = pd.Categorical(df_ns[category], categories=grade_order, ordered=True)
    dist = df_ns[category].value_counts().reindex(grade_order).fillna(0).astype(int)
    esg_heatmap_non_semi.append(dist.values)

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('반도체 (E/S/G)', '비반도체 (E/S/G)'),
    specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}]]
)

fig3.add_trace(
    go.Heatmap(
        z=esg_heatmap_semi,
        y=['환경', '사회', '지배구조'],
        x=grade_order,
        colorscale='Blues',
        text=esg_heatmap_semi,
        texttemplate='%{text}',
        textfont={"size": 12},
        showscale=False,
        name='반도체'
    ),
    row=1, col=1
)

fig3.add_trace(
    go.Heatmap(
        z=esg_heatmap_non_semi,
        y=['환경', '사회', '지배구조'],
        x=grade_order,
        colorscale='Oranges',
        text=esg_heatmap_non_semi,
        texttemplate='%{text}',
        textfont={"size": 12},
        showscale=False,
        name='비반도체'
    ),
    row=1, col=2
)

fig3.update_layout(
    title_text='<b>[EDA 4] E/S/G 세부 등급 분포 비교</b>',
    font=dict(family='Arial'),
    height=500,
    width=1000
)
fig3.show()

print("✅ 차트 3 완료")

✅ 차트 3 완료


In [ ]:
# [CELL 15] EDA 5: 산업코드별 분석
# ====================================================================

print("\n" + "="*70)
print("[EDA 5] 고객 세그먼트: 산업코드별 기업수 및 등급 분포")
print("="*70)

# 산업코드-산업명 매핑 딕셔너리 정의
industry_code_map = {
    'C2629': '기타 전자 부품 제조업',
    'C2612': '다이오드, 트랜지스터 및 유사 반도체 소자 제조업',
    'C2622': '인쇄회로기판 및 전자부품 실장기판 제조업',
    'C28202': '운송장비용 이차전지 제조업',
    'C262': '전자 부품 제조업',
    'C264': '통신 및 방송장비 제조업',
    'C2611': '전자집적회로 제조업',
    'C265': '시계, 측정, 시험, 항해 및 기타 정밀기기 제조업',
    'C28302': '기타 가정용 전기기기 제조업',
    'C2642': '방송 및 무선 통신장비 제조업',
    'C26299': '그 외 기타 전자 부품 제조업',
    'C26291': '전자 축전기 제조업',
    'C261': '반도체 제조업',
    'C281': '전동기, 발전기 및 전기 변환·제어장치 제조업',
}

# df_segment에 산업명 컬럼 추가 (매핑되지 않은 코드는 '기타/미분류'로 처리)
df_segment['산업명'] = df_segment['산업코드'].astype(str).map(industry_code_map).fillna('기타/미분류')

# 산업코드 및 산업명별 기업 수 집계
segment_by_industry_code = df_segment.groupby(['산업코드', '산업명']).size().reset_index(name='기업수')
segment_by_industry_code = segment_by_industry_code.sort_values('기업수', ascending=False)

print(f"\n산업코드별 기업 수 TOP 15:")
print(segment_by_industry_code.head(15).to_string(index=False))

# 시각화 1: 산업코드별 기업 수 분포 (상위 15개)
top_industry_codes = segment_by_industry_code.head(15).copy()
top_industry_codes['산업코드_이름'] = top_industry_codes['산업코드'] + ' (' + top_industry_codes['산업명'] + ')'

fig4 = px.bar(
    top_industry_codes,
    x='산업코드_이름',
    y='기업수',
    title='<b>[EDA 5] 산업코드별 기업 수 분포 (상위 15개)</b>',
    labels={'기업수': '기업 수 (개사)', '산업코드_이름': '산업코드 (산업명)'},
    color='기업수',
    color_continuous_scale='Viridis'
)
fig4.update_layout(
    font=dict(family='Arial'),
    height=500,
    width=900,
    xaxis={'tickangle': -45}
)
fig4.show()

print("✅ 차트 4 완료")


# 각 산업별 회사 목록 출력
print("\n" + "="*70)
print("[EDA 5] 각 산업별 회사 목록 (상위 5개 회사)")
print("="*70)

for _, row in top_industry_codes.iterrows():
    industry_code = row['산업코드']
    industry_name = row['산업명']
    print(f"\n【{industry_code} ({industry_name})】:")
    companies_in_industry = df_segment[df_segment['산업코드'] == industry_code].sort_values('기업명')
    if not companies_in_industry.empty:
        print(companies_in_industry[['기업명', 'ESG등급']].head(5).to_string(index=False))
    else:
        print("해당 산업에 속하는 기업이 없습니다.")


# 시각화 2: 산업코드별 ESG 등급 분포
print("\n" + "="*70)
print("[EDA 5] 산업코드별 ESG 등급 분포")
print("="*70)

df_top_industries = df_segment[df_segment['산업코드'].isin(top_industry_codes['산업코드'])].copy()
df_top_industries['산업코드_이름'] = df_top_industries['산업코드'] + ' (' + df_top_industries['산업명'] + ')'
df_top_industries['ESG등급'] = pd.Categorical(df_top_industries['ESG등급'], categories=grade_order, ordered=True)

industry_grade_dist = df_top_industries.groupby(['산업코드_이름', 'ESG등급']).size().reset_index(name='기업수')

fig_grade_dist = px.bar(
    industry_grade_dist,
    x='산업코드_이름',
    y='기업수',
    color='ESG등급',
    barmode='stack',
    title='<b>[EDA 5] 상위 산업코드별 ESG 등급 분포</b>',
    labels={'기업수': '기업 수 (개사)', '산업코드_이름': '산업코드 (산업명)', 'ESG등급': 'ESG 통합 등급'},
    category_orders={'ESG등급': grade_order},
    color_discrete_sequence=['#1D3557', '#457B9D', '#A8DADC', '#F4A261', '#E76F51', '#E63946']
)
fig_grade_dist.update_layout(
    font=dict(family='Arial'),
    height=600,
    width=1000,
    xaxis={'tickangle': -45}
)
fig_grade_dist.show()

print("✅ 추가 차트 완료")



[EDA 5] 고객 세그먼트: 산업코드별 기업수 및 등급 분포

산업코드별 기업 수 TOP 15:
  산업코드                          산업명  기업수
 C2629                 기타 전자 부품 제조업   12
 C2612  다이오드, 트랜지스터 및 유사 반도체 소자 제조업   10
 C2622       인쇄회로기판 및 전자부품 실장기판 제조업    8
C28202               운송장비용 이차전지 제조업    6
  C262                    전자 부품 제조업    6
  C264                통신 및 방송장비 제조업    5
   nan                       기타/미분류    4
 C2611                   전자집적회로 제조업    4
  C265 시계, 측정, 시험, 항해 및 기타 정밀기기 제조업    3
C28302              기타 가정용 전기기기 제조업    3
 C2642             방송 및 무선 통신장비 제조업    3
C26299             그 외 기타 전자 부품 제조업    3
C26291                   전자 축전기 제조업    3
  C261                      반도체 제조업    2
  C281    전동기, 발전기 및 전기 변환·제어장치 제조업    2


✅ 차트 4 완료

[EDA 5] 각 산업별 회사 목록 (상위 5개 회사)

【C2629 (기타 전자 부품 제조업)】:
       기업명 ESG등급
      KH바텍     A
      경인전자     D
    대주전자재료    B+
       드림텍     B
롯데에너지머티리얼즈    B+

【C2612 (다이오드, 트랜지스터 및 유사 반도체 소자 제조업)】:
       기업명 ESG등급
HD현대에너지솔루션     A
       KEC     D
     LB세미콘     C
     LX세미콘     A
    SK하이닉스     A

【C2622 (인쇄회로기판 및 전자부품 실장기판 제조업)】:
 기업명 ESG등급
대덕전자    B+
비에이치     B
삼성전기     A
  심텍     A
에이엔피     D

【C28202 (운송장비용 이차전지 제조업)】:
     기업명 ESG등급
 DN오토모티브     C
LG에너지솔루션     A
   더블유씨피     D
    세방전지     B
  에코프로비엠    B+

【C262 (전자 부품 제조업)】:
   기업명 ESG등급
 LG이노텍     A
덕산네오룩스     C
  써니전자     D
   이엠텍     D
 제이앤티씨     D

【C264 (통신 및 방송장비 제조업)】:
    기업명 ESG등급
   LG전자     A
  RFHIC     D
   대동전자     D
   삼성전자     A
주성코퍼레이션     D

【nan (기타/미분류)】:
                                                 기업명 ESG등급
*007340_디티알오토모티브' > 'DN오토모티브'는 동일 법인의 변경 전후 사명 (22년)   NaN
                                  *033180_KH필룩스_상장폐지   NaN
                                                 nan   NaN
         

/tmp/ipykernel_831/2549354073.py:85: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



✅ 추가 차트 완료


In [ ]:
# [CELL 16] EDA 6: 시가총액별 분석
# ====================================================================

print("\n" + "="*70)
print("[EDA 6] 고객 세그먼트: 시가총액별 분포 (2025년 기준)")
print("="*70)

def categorize_market_cap(value):
    if pd.isna(value):
        return '미분류'
    elif value >= 3_000_000_000_000:
        return '대형 (3조 이상)'
    elif value >= 1_000_000_000_000:
        return '중형 (1~3조)'
    elif value >= 300_000_000_000:
        return '소형 (3천억~1조)'
    else:
        return '초소형 (3천억 미만)'

df_segment['시가총액분류'] = df_segment['시가총액_수치'].apply(categorize_market_cap)

market_cap_order = ['대형 (3조 이상)', '중형 (1~3조)', '소형 (3천억~1조)', '초소형 (3천억 미만)', '미분류']
df_segment['시가총액분류'] = pd.Categorical(df_segment['시가총액분류'], categories=market_cap_order, ordered=True)
market_cap_dist = df_segment['시가총액분류'].value_counts().reindex(market_cap_order).fillna(0).reset_index()
market_cap_dist.columns = ['시가총액분류', '기업수']
market_cap_dist['비중(%)'] = (market_cap_dist['기업수'] / market_cap_dist['기업수'].sum() * 100).round(1)

print(f"\n시가총액 범위별 기업 분포:")
print(market_cap_dist.to_string(index=False))

# 시가총액 분류별 총 시가총액 계산
total_market_cap_per_segment = df_segment.groupby('시가총액분류', observed=False)['시가총액_수치'].sum().reset_index()
total_market_cap_per_segment.columns = ['시가총액분류', '총시가총액']

# 기존 market_cap_dist와 병합
market_cap_dist_combined = pd.merge(market_cap_dist, total_market_cap_per_segment, on='시가총액분류', how='left')

# 시가총액을 보기 쉽게 포맷하는 함수
def format_market_cap_display(value):
    if pd.isna(value) or value == 0:
        return 'N/A'
    elif value >= 1_000_000_000_000:
        return f"{value / 1_000_000_000_000:.1f}조원"
    elif value >= 100_000_000:
        return f"{value / 100_000_000:.1f}억원"
    else:
        return f"{value:.0f}원"

# 포맷된 총 시가총액 컬럼 추가
market_cap_dist_combined['총시가총액_표시'] = market_cap_dist_combined['총시가총액'].apply(format_market_cap_display)

# 시각화 1: 시가총액별 기업 분포 파이 차트
colors_marketcap = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#CCCCCC']

fig5 = px.pie(
    market_cap_dist_combined[market_cap_dist_combined['기업수'] > 0],
    names='시가총액분류', # 'labels' 대신 'names'를 사용하여 파이 슬라이스 이름 지정
    values='기업수',
    title='<b>[EDA 6-1] 시가총액별 기업 분포 (2025년 기준)</b>',
    color_discrete_sequence=colors_marketcap,
    hover_name='시가총액분류',
    hover_data={'기업수': True, '비중(%)': True, '총시가총액_표시': '총 시가총액'}, # 포맷된 총 시가총액을 툴팁에 추가
    labels={'시가총액분류': '시가총액 분류'} # 'labels' 인수에 딕셔너리 형태로 전달하여 오류 방지
)
fig5.update_traces(textposition='inside', textinfo='label+percent+value')
fig5.update_layout(
    font=dict(family='Arial'),
    height=600,
    width=800
)
fig5.show()

print("✅ 차트 5-1 완료")

# 시각화 2: 시가총액 분류별 ESG 등급 분포
print("\n" + "="*70)
print("[EDA 6-2] 시가총액 분류별 ESG 등급 분포")
print("="*70)

# '미분류' 시가총액을 제외한 데이터 필터링
df_filtered_for_esg = df_segment[df_segment['시가총액분류'] != '미분류'].copy()

# ESG 등급을 범주형으로 설정하여 정렬 순서 보장
df_filtered_for_esg['ESG등급'] = pd.Categorical(
    df_filtered_for_esg['ESG등급'],
    categories=grade_order,
    ordered=True
)

market_cap_esg_dist = df_filtered_for_esg.groupby(['시가총액분류', 'ESG등급'], observed=False).size().reset_index(name='기업수')

fig_market_cap_esg = px.bar(
    market_cap_esg_dist,
    x='시가총액분류',
    y='기업수',
    color='ESG등급',
    barmode='stack',
    title='<b>[EDA 6-2] 시가총액 분류별 ESG 등급 분포</b>',
    labels={'기업수': '기업 수 (개사)', '시가총액분류': '시가총액 분류', 'ESG등급': 'ESG 통합 등급'},
    category_orders={'시가총액분류': [mc for mc in market_cap_order if mc != '미분류'], 'ESG등급': grade_order},
    color_discrete_sequence=['#1D3557', '#457B9D', '#A8DADC', '#F4A261', '#E76F51', '#E63946']
)
fig_market_cap_esg.update_layout(
    font=dict(family='Arial'),
    height=600,
    width=900,
    xaxis={'tickangle': -45}
)
fig_market_cap_esg.show()

print("✅ 차트 5-2 완료")


[EDA 6] 고객 세그먼트: 시가총액별 분포 (2025년 기준)

시가총액 범위별 기업 분포:
      시가총액분류  기업수  비중(%)
  대형 (3조 이상)   18   18.8
   중형 (1~3조)   16   16.7
 소형 (3천억~1조)   27   28.1
초소형 (3천억 미만)   28   29.2
         미분류    7    7.3


✅ 차트 5-1 완료

[EDA 6-2] 시가총액 분류별 ESG 등급 분포


✅ 차트 5-2 완료


In [ ]:
# [CELL 17] EDA 7: 세그먼트별 ESG 등급 분포
# ====================================================================

print("\n" + "="*70)
print("[EDA 7] 세그먼트별 ESG 등급 분포: 시가총액 × 업종분류")
print("="*70)

df_segment_filtered = df_segment[df_segment['시가총액분류'] != '미분류'].copy()

for market_size in market_cap_order[:-1]:
    print(f"\n【{market_size}】")
    subset = df_segment_filtered[df_segment_filtered['시가총액분류'] == market_size]
    print(f"총 {len(subset)}개사 | 반도체 {len(subset[subset['업종분류']=='반도체'])}개사, 비반도체 {len(subset[subset['업종분류']=='비반도체'])}개사")

# 시각화
segment_matrix = df_segment_filtered.groupby(
    ['시가총액분류', '업종분류', 'ESG등급']
).size().reset_index(name='기업수')

segment_matrix_viz = segment_matrix.copy()
segment_matrix_viz['시가총액분류'] = pd.Categorical(
    segment_matrix_viz['시가총액분류'],
    categories=market_cap_order[:-1],
    ordered=True
)
segment_matrix_viz = segment_matrix_viz.sort_values('시가총액분류')

fig6 = px.bar(
    segment_matrix_viz,
    x='시가총액분류',
    y='기업수',
    color='ESG등급',
    barmode='stack',
    title='<b>[EDA 7] 시가총액별 ESG 등급 분포</b>',
    labels={'기업수': '기업 수 (개사)', '시가총액분류': '시가총액 범위'},
    color_discrete_sequence=['#1D3557', '#457B9D', '#A8DADC', '#F4A261', '#E76F51', '#E63946']
)
fig6.update_layout(
    font=dict(family='Arial'),
    xaxis={'categoryorder': 'array', 'categoryarray': market_cap_order[:-1]},
    height=500,
    width=900
)
fig6.show()

print("✅ 차트 6 완료")


[EDA 7] 세그먼트별 ESG 등급 분포: 시가총액 × 업종분류

【대형 (3조 이상)】
총 18개사 | 반도체 1개사, 비반도체 17개사

【중형 (1~3조)】
총 16개사 | 반도체 2개사, 비반도체 14개사

【소형 (3천억~1조)】
총 27개사 | 반도체 7개사, 비반도체 20개사

【초소형 (3천억 미만)】
총 28개사 | 반도체 5개사, 비반도체 23개사


/tmp/ipykernel_831/3691699327.py:16: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



✅ 차트 6 완료


In [ ]:
# [CELL 18] 최종 세그먼트 요약
# ====================================================================

print("\n" + "="*70)
print("[최종 세그먼트 요약] 시가총액 × 업종 매트릭스")
print("="*70)

segment_summary = df_segment_filtered.groupby(
    ['시가총액분류', '업종분류']
).size().reset_index(name='기업수')

segment_summary_pivot = segment_summary.pivot(
    index='시가총액분류',
    columns='업종분류',
    values='기업수'
).fillna(0).astype(int)

segment_summary_pivot = segment_summary_pivot.reindex(market_cap_order[:-1])
segment_summary_pivot['합계'] = segment_summary_pivot.sum(axis=1)

print("\n시가총액 × 업종분류 세그먼트 기업수:")
print(segment_summary_pivot)
print(f"\n총합: {segment_summary_pivot['합계'].sum()}개사")

# 시각화
fig7 = go.Figure(data=go.Heatmap(
    z=segment_summary_pivot.iloc[:, :-1].values,
    x=segment_summary_pivot.columns[:-1],
    y=segment_summary_pivot.index,
    text=segment_summary_pivot.iloc[:, :-1].values,
    texttemplate='%{text}개사',
    textfont={"size": 14},
    colorscale='YlOrRd',
    showscale=True
))

fig7.update_layout(
    title='<b>[세그먼트] 시가총액 × 업종 분류 기업수 히트맵</b>',
    xaxis_title='업종분류',
    yaxis_title='시가총액 분류',
    font=dict(family='Arial'),
    height=500,
    width=700
)
fig7.show()

print("✅ 차트 7 완료")


[최종 세그먼트 요약] 시가총액 × 업종 매트릭스

시가총액 × 업종분류 세그먼트 기업수:
업종분류          반도체  비반도체  합계
시가총액분류                     
대형 (3조 이상)      1    17  18
중형 (1~3조)       2    14  16
소형 (3천억~1조)     7    20  27
초소형 (3천억 미만)    5    23  28

총합: 89개사


/tmp/ipykernel_831/1158588359.py:8: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



✅ 차트 7 완료


In [ ]:
# [CELL 19] 세그먼트별 기업 리스트 (상세)
# ====================================================================

print("\n" + "="*70)
print("[참고] 주요 세그먼트별 기업 예시")
print("="*70)

# Segment A: 플래그십 (대형 3조↑)
print("\n【Segment A: 플래그십 세그먼트 (대형 3조↑)】")
segment_a = df_segment_filtered[
    (df_segment_filtered['시가총액분류'] == '대형 (3조 이상)')
].sort_values('ESG등급')

print(f"총 {len(segment_a)}개사")
print(segment_a[['기업명', '업종분류', 'ESG등급', '환경', '사회', '지배구조']].head(10).to_string(index=False))

# Segment B: 성장가능 (중형 1~3조)
print("\n【Segment B: 성장 가능 세그먼트 (중형 1~3조)】")
segment_b = df_segment_filtered[
    (df_segment_filtered['시가총액분류'] == '중형 (1~3조)')
].sort_values('ESG등급')

print(f"총 {len(segment_b)}개사")
print(segment_b[['기업명', '업종분류', 'ESG등급', '환경', '사회', '지배구조']].head(10).to_string(index=False))

print("\n" + "="*70)
print("✅ EDA 완료!")
print("="*70)


[참고] 주요 세그먼트별 기업 예시

【Segment A: 플래그십 세그먼트 (대형 3조↑)】
총 18개사
     기업명 업종분류 ESG등급 환경 사회 지배구조
HD현대일렉트릭 비반도체    A+ A+ A+    A
  SK하이닉스  반도체     A  A A+   B+
LG에너지솔루션 비반도체     A B+ A+    A
 LG디스플레이 비반도체     A  A A+   B+
   한화시스템 비반도체     A  A A+   B+
  포스코퓨처엠 비반도체     A  A A+    A
   LG이노텍 비반도체     A  A A+    A
 엘에스일렉트릭 비반도체     A A+  A    A
   삼성SDI 비반도체     A  A A+   B+
    삼성전자 비반도체     A B+ A+   B+

【Segment B: 성장 가능 세그먼트 (중형 1~3조)】
총 16개사
       기업명 업종분류 ESG등급 환경 사회 지배구조
     두산퓨얼셀 비반도체     A  A A+   B+
        심텍 비반도체     A  A A+   B+
SK아이이테크놀로지 비반도체     A  A A+   B+
     DB하이텍  반도체    B+ B+  A   B+
      가온전선 비반도체    B+  A A+    B
      대덕전자 비반도체    B+ B+ A+    B
롯데에너지머티리얼즈 비반도체    B+ B+ B+    B
    하나마이크론  반도체     C B+  A    D
     서진시스템 비반도체     C  C  B    D
    에스앤에스텍 비반도체     C  D  D   B+

✅ EDA 완료!


In [ ]:
# [CELL 20] 데이터 엑셀로 저장 (선택사항)
# ====================================================================

# Colab에서 다운로드할 수 있도록 저장
from google.colab import files

# 최종 세그먼트 데이터 저장
output_excel = '/tmp/segment_classification_results.xlsx'

with pd.ExcelWriter(output_excel) as writer:
    df_segment_filtered.to_excel(writer, sheet_name='전체데이터', index=False)
    segment_summary_pivot.to_excel(writer, sheet_name='세그먼트요약')
    grade_semi.to_excel(writer, sheet_name='반도체등급분포', index=False)
    grade_non_semi.to_excel(writer, sheet_name='비반도체등급분포', index=False)

print(f"✅ 결과 엑셀 저장됨: {output_excel}")
print("\n다운로드:")
files.download(output_excel)

print("\n✅ 모든 작업 완료!")

✅ 결과 엑셀 저장됨: /tmp/segment_classification_results.xlsx

다운로드:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ 모든 작업 완료!


In [ ]:
# [CELL 19] EDA 5-1: 산업코드별 기업 목록 (등급순) 및 등급 분포 시각화
# ====================================================================

print("\n" + "="*70)
print("[EDA 5-1] 산업코드별 기업 목록 (등급순 정렬) 및 ESG 등급 분포")
print("="*70)

# 상위 5개 산업코드 추출
top_5_codes = segment_by_industry_code.head(5)['산업코드'].tolist()

for idx, code in enumerate(top_5_codes, 1):
    print(f"\n【{idx}. 산업코드: {code}】")

    companies = df_segment[df_segment['산업코드'] == code].sort_values('ESG등급')
    count = len(companies)

    # 등급별 카운트
    grade_counts = companies['ESG등급'].value_counts()
    grade_str = " | ".join([f"{g}등급: {grade_counts.get(g, 0)}" for g in grade_order if grade_counts.get(g, 0) > 0])

    print(f"총 {count}개사 | {grade_str}")
    print("\n" + "-"*100)
    print(companies[['기업명', 'ESG등급', '환경', '사회', '지배구조', '시가총액']].to_string(index=False))
    print("-"*100)

    # 각 산업코드별 ESG 등급 분포 시각화
    print(f"\n[시각화] {code} - ESG 등급 분포")
    grade_dist_df = grade_counts.reindex(grade_order).fillna(0).reset_index(name='기업수')
    grade_dist_df.columns = ['ESG등급', '기업수']

    fig_industry_grade = px.bar(
        grade_dist_df,
        x='ESG등급',
        y='기업수',
        title=f'<b>{idx}. 산업코드: {code} - ESG 등급 분포</b>',
        labels={'기업수': '기업 수 (개사)', 'ESG등급': 'ESG 통합 등급'},
        color='ESG등급',
        color_discrete_sequence=['#1D3557', '#457B9D', '#A8DADC', '#F4A261', '#E76F51', '#E63946'] # 기존 차트와 색상 통일
    )
    fig_industry_grade.update_layout(
        font=dict(family='Arial'),
        height=400,
        width=700,
        xaxis={'categoryorder': 'array', 'categoryarray': grade_order}
    )
    fig_industry_grade.show()
    print("\n" + "="*70)


[EDA 5-1] 산업코드별 기업 목록 (등급순 정렬) 및 ESG 등급 분포

【1. 산업코드: C2629】
총 12개사 | A등급: 1 | B+등급: 4 | B등급: 1 | C등급: 3 | D등급: 3

----------------------------------------------------------------------------------------------------
       기업명 ESG등급 환경 사회 지배구조         시가총액
      KH바텍     A  A A+    A          NaN
    대주전자재료    B+ B+  A    B 9.830180e+11
롯데에너지머티리얼즈    B+ B+ B+    B 1.599760e+12
   아이티엠반도체    B+ B+  A    B 2.218390e+11
   이녹스첨단소재    B+ B+ B+    B 4.533540e+11
       드림텍     B  B  B   B+ 4.863640e+11
       솔루엠     C  B B+    D 7.932710e+11
    에스앤에스텍     C  D  D   B+ 1.030470e+12
     에프에스티     C  C  B    D 6.729620e+11
      경인전자     D  D  D    D 3.220134e+10
      리노공업     D  D  D    C 4.595570e+12
       파트론     D  C  D    D 3.833500e+11
----------------------------------------------------------------------------------------------------

[시각화] C2629 - ESG 등급 분포




【2. 산업코드: C2612】
총 10개사 | A등급: 3 | B등급: 1 | C등급: 3 | D등급: 3

----------------------------------------------------------------------------------------------------
       기업명 ESG등급 환경 사회 지배구조         시가총액
HD현대에너지솔루션     A  A  A    A 6.126400e+11
     LX세미콘     A  A A+   B+ 7.993900e+11
    SK하이닉스     A  A A+   B+ 4.739295e+14
   하나머티리얼즈     B B+ A+    D 8.524180e+11
     서울반도체     C  C  B    D 3.620770e+11
     신성이엔지     C  C  B    B 3.270930e+11
     LB세미콘     C B+  A    D          NaN
       광전자     D  C  D    D 1.044150e+11
     시그네틱스     D  C  C    D 6.703955e+10
       KEC     D  C  D    D 1.397310e+11
----------------------------------------------------------------------------------------------------

[시각화] C2612 - ESG 등급 분포




【3. 산업코드: C2622】
총 8개사 | A등급: 2 | B+등급: 1 | B등급: 2 | C등급: 2 | D등급: 1

----------------------------------------------------------------------------------------------------
   기업명 ESG등급 환경 사회 지배구조         시가총액
  삼성전기     A  A A+   B+ 1.904690e+13
    심텍     A  A A+   B+ 1.842840e+12
  대덕전자    B+ B+ A+    B 2.327540e+12
  비에이치     B B+ A+    D 5.427560e+11
이수페타시스     B B+ A+    D 8.750380e+12
 인터플렉스     C  B  B    C 2.468050e+11
코리아써키트     C  B B+    D 1.140880e+12
  에이엔피     D  C  D    D 1.940026e+10
----------------------------------------------------------------------------------------------------

[시각화] C2622 - ESG 등급 분포




【4. 산업코드: C28202】
총 6개사 | A등급: 1 | B+등급: 2 | B등급: 1 | C등급: 1 | D등급: 1

----------------------------------------------------------------------------------------------------
     기업명 ESG등급 환경 사회 지배구조         시가총액
LG에너지솔루션     A B+ A+    A 8.622900e+13
  에코프로비엠    B+  B  A    B 1.433770e+13
    엘앤에프    B+ B+  A    B 3.827930e+12
    세방전지     B  A  A    D 8.792000e+11
 DN오토모티브     C  B B+    D 1.427670e+12
   더블유씨피     D  C  D    D 2.111810e+11
----------------------------------------------------------------------------------------------------

[시각화] C28202 - ESG 등급 분포




【5. 산업코드: C262】
총 6개사 | A등급: 2 | C등급: 1 | D등급: 3

----------------------------------------------------------------------------------------------------
   기업명 ESG등급 환경 사회 지배구조         시가총액
 해성디에스     A  A  A   B+ 9.469000e+11
 LG이노텍     A  A A+    A 6.413790e+12
덕산네오룩스     C  B  A    D 9.063380e+11
  써니전자     D  C  D    D 5.539160e+10
   이엠텍     D  D  D    C 1.386050e+11
 제이앤티씨     D  C  D    D 1.146560e+12
----------------------------------------------------------------------------------------------------

[시각화] C262 - ESG 등급 분포


In [ ]:
# [CELL 20] 데이터 준비: 보고서 발행 유형 생성
# ====================================================================

# 보고서 발행 유형 컬럼 생성 (이미 있으면 패스)
if '보고서_발행_유형' not in df_segment.columns:
    report_type_list = []
    for idx, row in df_segment.iterrows():
        if row['지속가능경영보고서_발행'] == 1 and row['기업지배구조보고서_발행'] == 1:
            report_type_list.append('둘 다 발행')
        elif row['지속가능경영보고서_발행'] == 1:
            report_type_list.append('지속가능경영')
        elif row['기업지배구조보고서_발행'] == 1:
            report_type_list.append('지배구조')
        else:
            report_type_list.append('미발행')

    df_segment['보고서_발행_유형'] = report_type_list
    df_segment_filtered['보고서_발행_유형'] = df_segment_filtered['산업코드'].apply(
        lambda x: df_segment[df_segment['산업코드'] == x]['보고서_발행_유형'].iloc[0]
        if len(df_segment[df_segment['산업코드'] == x]) > 0 else '미발행'
    )

print("✅ 보고서 발행 유형 데이터 준비 완료")

✅ 보고서 발행 유형 데이터 준비 완료


In [ ]:
# [CELL 21] EDA 5-2: 산업코드별 시가총액 분석
# ====================================================================

print("\n" + "="*70)
print("[EDA 5-2] 산업코드별 시가총액 분석")
print("="*70)

# 산업코드별 시가총액 통계
market_cap_stats = df_segment.groupby('산업코드')['시가총액_수치'].describe().reset_index()
market_cap_stats.columns = ['산업코드', 'count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']

# 기업수와 합계 추가
market_cap_stats['합계'] = df_segment.groupby('산업코드')['시가총액_수치'].sum().values

# 조 단위로 표시
def format_trillion(x):
    if pd.isna(x) or x == 0:
        return 'N/A'
    return f"₩{x/1e12:.1f}조"

print("\n산업코드별 시가총액 통계 (상위 15개):")
display_cols = ['산업코드', 'count', 'mean', '50%', 'max', '합계']
market_cap_display = market_cap_stats[display_cols].head(15).copy()
market_cap_display.columns = ['산업코드', '기업수', '평균', '중간값', '최대', '합계']

for col in ['평균', '중간값', '최대', '합계']:
    market_cap_display[col] = market_cap_display[col].apply(format_trillion)

print(market_cap_display.to_string(index=False))

# 시각화
market_cap_viz = df_segment.groupby('산업코드').agg({
    '시가총액_수치': ['mean', 'count']
}).reset_index()
market_cap_viz.columns = ['산업코드', '평균시가총액', '기업수']
market_cap_viz = market_cap_viz.sort_values('평균시가총액', ascending=False).head(15)
market_cap_viz['평균시총_조'] = market_cap_viz['평균시가총액'] / 1e12

fig_marketcap = px.bar(
    market_cap_viz,
    x='산업코드',
    y='평균시총_조',
    # size='기업수', # px.bar에서는 'size' 인수를 지원하지 않으므로 제거
    title='<b>[EDA 5-2] 산업코드별 평균 시가총액 (상위 15개, 버블 크기=기업수)</b>',
    labels={'평균시총_조': '평균 시가총액 (조원)', '산업코드': '산업코드'},
    color='평균시총_조',
    color_continuous_scale='Viridis',
    hover_data=['기업수']
)
fig_marketcap.update_layout(
    font=dict(family='Arial'),
    height=500,
    width=1000,
    xaxis={'tickangle': -45}
)
fig_marketcap.show()

print("\n✅ 차트 생성 완료: 산업코드별 시가총액")


[EDA 5-2] 산업코드별 시가총액 분석

산업코드별 시가총액 통계 (상위 15개):
  산업코드  기업수      평균   중간값      최대      합계
  C261  2.0   ₩0.2조 ₩0.2조   ₩0.4조   ₩0.4조
 C2611  4.0   ₩1.4조 ₩1.2조   ₩2.9조   ₩5.6조
 C2612  9.0  ₩53.0조 ₩0.4조 ₩473.9조 ₩477.2조
  C262  6.0   ₩1.6조 ₩0.9조   ₩6.4조   ₩9.6조
 C2621  1.0   ₩5.9조 ₩5.9조   ₩5.9조   ₩5.9조
 C2622  8.0   ₩4.2조 ₩1.5조  ₩19.0조  ₩33.9조
 C2629 11.0   ₩1.0조 ₩0.7조   ₩4.6조  ₩11.3조
C26291  3.0   ₩0.2조 ₩0.2조   ₩0.3조   ₩0.5조
C26292  0.0     N/A   N/A     N/A     N/A
C26299  3.0   ₩3.7조 ₩0.5조  ₩10.3조  ₩11.0조
  C263  1.0   ₩0.5조 ₩0.5조   ₩0.5조   ₩0.5조
C26310  1.0   ₩0.0조 ₩0.0조   ₩0.0조   ₩0.0조
  C264  5.0 ₩145.2조 ₩0.9조 ₩709.8조 ₩725.8조
 C2642  3.0   ₩0.3조 ₩0.3조   ₩0.6조   ₩1.0조
C26429  1.0   ₩0.6조 ₩0.6조   ₩0.6조   ₩0.6조



✅ 차트 생성 완료: 산업코드별 시가총액


In [ ]:
# [CELL 22] EDA 5-3: 산업코드별 보고서 발행 현황
# ====================================================================

print("\n" + "="*70)
print("[EDA 5-3] 산업코드별 보고서 발행 현황")
print("="*70)

# 보고서 발행 유형 집계
report_by_code = df_segment.groupby(['산업코드', '보고서_발행_유형']).size().reset_index(name='기업수')
report_pivot = report_by_code.pivot(index='산업코드', columns='보고서_발행_유형', values='기업수').fillna(0).astype(int)
report_pivot['합계'] = report_pivot.sum(axis=1)
report_pivot = report_pivot.sort_values('합계', ascending=False).head(10)

print("\n산업코드별 보고서 발행 유형 (상위 10개):")
print(report_pivot)

# 발행률 계산
if '미발행' in report_pivot.columns:
    report_pivot['발행률(%)'] = (
        (report_pivot['합계'] - report_pivot['미발행']) / report_pivot['합계'] * 100
    ).round(1)
else:
    report_pivot['발행률(%)'] = 100.0

print("\n발행률:")
print(report_pivot[['합계', '발행률(%)']])

# 시각화
report_by_code_top = report_by_code[report_by_code['산업코드'].isin(report_pivot.index)]

fig_report = px.bar(
    report_by_code_top,
    x='산업코드',
    y='기업수',
    color='보고서_발행_유형',
    barmode='stack',
    title='<b>[EDA 5-3] 산업코드별 보고서 발행 현황 (상위 10개)</b>',
    labels={'기업수': '기업 수 (개사)', '산업코드': '산업코드'},
    color_discrete_map={
        '미발행': '#E63946',
        '지속가능경영': '#F4A261',
        '지배구조': '#A8DADC',
        '둘 다 발행': '#1D3557'
    }
)
fig_report.update_layout(
    font=dict(family='Arial'),
    height=500,
    width=1000,
    xaxis={'tickangle': -45}
)
fig_report.show()

print("\n✅ 차트 생성 완료: 산업코드별 보고서 발행")


[EDA 5-3] 산업코드별 보고서 발행 현황

산업코드별 보고서 발행 유형 (상위 10개):
보고서_발행_유형  둘 다 발행  미발행  지배구조만  지속가능경영만  합계
산업코드                                      
C2629           3    7      0        2  12
C2612           4    4      0        2  10
C2622           4    3      1        0   8
C262            2    3      0        1   6
C28202          4    1      0        1   6
C264            2    3      0        0   5
C2611           1    2      0        1   4
C26299          1    0      2        0   3
C2642           0    3      0        0   3
C265            0    3      0        0   3

발행률:
보고서_발행_유형  합계  발행률(%)
산업코드                 
C2629      12    41.7
C2612      10    60.0
C2622       8    62.5
C262        6    50.0
C28202      6    83.3
C264        5    40.0
C2611       4    50.0
C26299      3   100.0
C2642       3     0.0
C265        3     0.0


KeyboardInterrupt: 

In [ ]:
# [CELL 22] EDA 5-3: 산업코드별 보고서 발행 현황
# ====================================================================

print("\n" + "="*70)
print("[EDA 5-3] 산업코드별 보고서 발행 현황")
print("="*70)

# 보고서 발행 유형 집계
report_by_code = df_segment.groupby(['산업코드', '보고서_발행_유형']).size().reset_index(name='기업수')
report_pivot = report_by_code.pivot(index='산업코드', columns='보고서_발행_유형', values='기업수').fillna(0).astype(int)
report_pivot['합계'] = report_pivot.sum(axis=1)
report_pivot = report_pivot.sort_values('합계', ascending=False).head(10)

print("\n산업코드별 보고서 발행 유형 (상위 10개):")
print(report_pivot)

# 발행률 계산
if '미발행' in report_pivot.columns:
    report_pivot['발행률(%)'] = (
        (report_pivot['합계'] - report_pivot['미발행']) / report_pivot['합계'] * 100
    ).round(1)
else:
    report_pivot['발행률(%)'] = 100.0

print("\n발행률:")
print(report_pivot[['합계', '발행률(%)']])

# 시각화
report_by_code_top = report_by_code[report_by_code['산업코드'].isin(report_pivot.index)]

fig_report = px.bar(
    report_by_code_top,
    x='산업코드',
    y='기업수',
    color='보고서_발행_유형',
    barmode='stack',
    title='<b>[EDA 5-3] 산업코드별 보고서 발행 현황 (상위 10개)</b>',
    labels={'기업수': '기업 수 (개사)', '산업코드': '산업코드'},
    color_discrete_map={
        '미발행': '#E63946',
        '지속가능경영': '#F4A261',
        '지배구조': '#A8DADC',
        '둘 다 발행': '#1D3557'
    }
)
fig_report.update_layout(
    font=dict(family='Arial'),
    height=500,
    width=1000,
    xaxis={'tickangle': -45}
)
fig_report.show()

print("\n✅ 차트 생성 완료: 산업코드별 보고서 발행")

In [ ]:
# [CELL 24] 시각화: Segment별 등급 분포 히트맵
# ====================================================================

# 히트맵용 데이터 준비
segment_heatmap_data = []
segment_labels = []

for market_cap in market_cap_order[:-1]:
    for industry in ['반도체', '비반도체']:
        subset = df_segment_filtered[
            (df_segment_filtered['시가총액분류'] == market_cap) &
            (df_segment_filtered['업종분류'] == industry)
        ]

        if len(subset) > 0:
            row = []
            for grade in grade_order:
                count = len(subset[subset['ESG등급'] == grade])
                row.append(count)
            segment_heatmap_data.append(row)
            segment_labels.append(f"{market_cap}\n({industry})")

fig_segment_heatmap = go.Figure(data=go.Heatmap(
    z=segment_heatmap_data,
    x=grade_order,
    y=segment_labels,
    text=segment_heatmap_data,
    texttemplate='%{text}개사',
    textfont={"size": 12},
    colorscale='RdYlGn',
    showscale=True,
    colorbar=dict(title="기업수")
))

fig_segment_heatmap.update_layout(
    title='<b>[EDA 6-1] Segment별 ESG 등급 분포 히트맵</b>',
    xaxis_title='ESG 등급',
    yaxis_title='Segment',
    font=dict(family='Arial'),
    height=600,
    width=800
)
fig_segment_heatmap.show()

print("\n✅ 차트 생성 완료: Segment별 등급 분포 히트맵")

In [ ]:
# [CELL 25] EDA 6-2: Segment별 E/S/G 프로필
# ====================================================================

print("\n" + "="*70)
print("[EDA 6-2] Segment별 평균 E/S/G 프로필")
print("="*70)

# E/S/G를 숫자로 변환 (A+=6, A=5, B+=4, B=3, C=2, D=1)
grade_map = {'A+': 6, 'A': 5, 'B+': 4, 'B': 3, 'C': 2, 'D': 1}

df_segment_filtered_copy = df_segment_filtered.copy()
df_segment_filtered_copy['환경_수'] = df_segment_filtered_copy['환경'].map(grade_map)
df_segment_filtered_copy['사회_수'] = df_segment_filtered_copy['사회'].map(grade_map)
df_segment_filtered_copy['지배구조_수'] = df_segment_filtered_copy['지배구조'].map(grade_map)

esg_by_segment = df_segment_filtered_copy.groupby(
    ['시가총액분류', '업종분류']
).agg({
    '환경_수': 'mean',
    '사회_수': 'mean',
    '지배구조_수': 'mean',
    '기업명': 'count'
}).round(2).reset_index()

esg_by_segment.columns = ['시가총액분류', '업종분류', '환경_평균', '사회_평균', '지배구조_평균', '기업수']

# 숫자를 다시 등급으로 변환 (표시용)
def num_to_grade(num):
    if pd.isna(num):
        return 'N/A'
    elif num < 1.5:
        return 'D'
    elif num < 2.5:
        return 'C'
    elif num < 3.5:
        return 'B'
    elif num < 4.5:
        return 'B+'
    elif num < 5.5:
        return 'A'
    else:
        return 'A+'

esg_by_segment['환경_등급'] = esg_by_segment['환경_평균'].apply(num_to_grade)
esg_by_segment['사회_등급'] = esg_by_segment['사회_평균'].apply(num_to_grade)
esg_by_segment['지배구조_등급'] = esg_by_segment['지배구조_평균'].apply(num_to_grade)

print("\nSegment별 평균 E/S/G 등급:")
print(esg_by_segment[['시가총액분류', '업종분류', '환경_등급', '사회_등급', '지배구조_등급', '기업수']].to_string(index=False))

# 시각화
esg_by_segment_melted = esg_by_segment.melt(
    id_vars=['시가총액분류', '업종분류', '기업수'],
    value_vars=['환경_평균', '사회_평균', '지배구조_평균'],
    var_name='항목',
    value_name='평균점수'
)

esg_by_segment_melted['Segment'] = (
    esg_by_segment_melted['시가총액분류'].str[:10] + '\n' + esg_by_segment_melted['업종분류']
)

esg_by_segment_melted['항목'] = esg_by_segment_melted['항목'].map({
    '환경_평균': '환경(E)',
    '사회_평균': '사회(S)',
    '지배구조_평균': '지배구조(G)'
})

fig_esg_profile = px.bar(
    esg_by_segment_melted,
    x='Segment',
    y='평균점수',
    color='항목',
    barmode='group',
    title='<b>[EDA 6-2] Segment별 평균 E/S/G 등급 프로필</b>',
    labels={'평균점수': '평균 점수 (1=D, 6=A+)', 'Segment': 'Segment'},
    color_discrete_map={'환경(E)': '#2E86AB', '사회(S)': '#A23B72', '지배구조(G)': '#F18F01'},
    height=500,
    width=1100
)
fig_esg_profile.update_layout(
    font=dict(family='Arial'),
    xaxis={'tickangle': -45},
    yaxis_range=[0, 6] # yaxis_range를 update_layout 내에서 설정
)
fig_esg_profile.show()

print("\n✅ 차트 생성 완료: Segment별 E/S/G 프로필")

In [ ]:
# [CELL 26] EDA 6-3: Segment별 보고서 발행 현황
# ====================================================================

print("\n" + "="*70)
print("[EDA 6-3] Segment별 보고서 발행 현황")
print("="*70)

report_by_segment = df_segment_filtered.groupby(
    ['시가총액분류', '업종분류', '보고서_발행_유형']
).size().reset_index(name='기업수')

report_by_segment_pivot = report_by_segment.pivot_table(
    index=['시가총액분류', '업종분류'],
    columns='보고서_발행_유형',
    values='기업수',
    fill_value=0
).astype(int)

report_by_segment_pivot['총기업수'] = report_by_segment_pivot.sum(axis=1)

# 발행률 계산
if '미발행' in report_by_segment_pivot.columns:
    report_by_segment_pivot['발행률(%)'] = (
        (report_by_segment_pivot['총기업수'] - report_by_segment_pivot['미발행']) /
        report_by_segment_pivot['총기업수'] * 100
    ).round(1)
else:
    report_by_segment_pivot['발행률(%)'] = 100.0

print("\nSegment별 보고서 발행 현황:")
print(report_by_segment_pivot[['총기업수', '발행률(%)']])

# 시각화
report_by_segment_melted = report_by_segment.copy()
report_by_segment_melted['Segment'] = (
    report_by_segment_melted['시가총액분류'].str[:10] + '\n' + report_by_segment_melted['업종분류']
)

fig_report_segment = px.bar(
    report_by_segment_melted,
    x='Segment',
    y='기업수',
    color='보고서_발행_유형',
    barmode='stack',
    title='<b>[EDA 6-3] Segment별 보고서 발행 현황</b>',
    labels={'기업수': '기업 수', '보고서_발행_유형': '보고서 유형'},
    color_discrete_map={
        '미발행': '#E63946',
        '지속가능경영': '#F4A261',
        '지배구조': '#A8DADC',
        '둘 다 발행': '#1D3557'
    }
)
fig_report_segment.update_layout(
    font=dict(family='Arial'),
    height=500,
    width=1100,
    xaxis={'tickangle': -45}
)
fig_report_segment.show()

print("\n✅ 차트 생성 완료: Segment별 보고서 발행")

# 현재까지의 발견:
1️⃣ 반도체(15개사) vs 비반도체(76개사) - 극단적 차이

2️⃣ 지배구조(G)가 업계 공통 약점 (D등급 40~56%)

3️⃣ 시가총액이 가장 실용적인 세그먼트 기준

4️⃣ Segment A(18) > B(16) > C(27) > D(28)

# **[STEP 2] 세그먼트 분류**
- 벤치마킹 대상 선정 (각 세그먼트별 1-2개사)
- 컨설팅 후보 선정 (각 세그먼트별 3-5개사)
- 산업코드 드릴다운 분석

In [ ]:
# 📌 세그먼트별 전략:

# 【Segment A: 플래그십 (18개사)】
# ├─ 컨설팅 전략: 💰 딥다이브 고비용
# ├─ 세일즈 포인트: 공급망 ESG (고객사: 삼성·LG·SK·현대차)
# └─ 우선순위: ⭐⭐⭐⭐⭐ (최우선)

# 【Segment B: 성장 가능 (16개사)】
# ├─ 컨설팅 전략: 📋 선별형 (우수 vs 개선 분리)
# ├─ 세일즈 포인트: 예산 효율화, 공급망 의존성
# └─ 우선순위: ⭐⭐⭐⭐ (높음)

# 【Segment C: 기술 특화 (27개사)】
# ├─ 컨설팅 전략: 🔧 산업별 맞춤형
# ├─ 세일즈 포인트: 협력사 지위, 우수 사례
# └─ 우선순위: ⭐⭐⭐ (중간)

# 【Segment D: 표준화 (28개사)】
# ├─ 컨설팅 전략: 📱 저비용 스케일 모델
# ├─ 세일즈 포인트: 자가진단 도구, 뉴스레터
# └─ 우선순위: ⭐⭐ (낮음)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 🎯 3단계 로드맵:

# [STEP 1] 기초 EDA ✅ (완료)
# ├─ 반도체 vs 비반도체 분석
# ├─ ESG 등급 분포
# ├─ 산업코드별 분석
# ├─ 시가총액별 분석
# └─ Segment별 분석

# [STEP 2] 세그먼트 분류 (진행 예정)
# ├─ 벤치마킹 대상 선정 (각 세그먼트별 1-2개사)
# ├─ 컨설팅 후보 선정 (각 세그먼트별 3-5개사)
# └─ 산업코드 드릴다운 분석

# [STEP 3] 세부 후보군 정하기 (최종)
# ├─ Segment별 우선순위 정렬
# ├─ 기업별 특성 분석
# └─ 최종 타겟팅 전략 수립

In [ ]:
# [CELL 35] 비반도체 데이터 필터링
# ====================================================================

print("\n" + "="*100)
print("[비반도체 세그먼트 분류] 2×2 매트릭스 준비")
print("="*100)

# 비반도체만 필터링
df_non_semi_all = df_segment_filtered[df_segment_filtered['업종분류'] == '비반도체'].copy()

print(f"\n📊 분석 대상: 비반도체 {len(df_non_semi_all)}개사")
print(f"시가총액 범위: {df_non_semi_all['시가총액_수치'].min()/1e12:.1f}조 ~ {df_non_semi_all['시가총액_수치'].max()/1e12:.1f}조")

# 시가총액 기준: 1조 이상 vs 미만
threshold_marketcap = 1_000_000_000_000  # 1조

df_non_semi_all['규모분류'] = df_non_semi_all['시가총액_수치'].apply(
    lambda x: '대형 (1조↑)' if x >= threshold_marketcap else '중소 (1조↓)'
)

# ESG등급 기준: A/B+ vs C/D
df_non_semi_all['등급분류'] = df_non_semi_all['ESG등급'].apply(
    lambda x: '우수 (A/B+)' if x in ['A+', 'A', 'B+'] else '부실 (C/D)'
)

# 세그먼트 라벨
def assign_segment_2x2(row):
    if row['규모분류'] == '대형 (1조↑)' and row['등급분류'] == '우수 (A/B+)':
        return 'I-A (플래그십)'
    elif row['규모분류'] == '대형 (1조↑)' and row['등급분류'] == '부실 (C/D)':
        return 'I-B (대형개선사)'
    elif row['규모분류'] == '중소 (1조↓)' and row['등급분류'] == '우수 (A/B+)':
        return 'II-A (성장리더)'
    else:
        return 'II-B (표준화)'

df_non_semi_all['Segment'] = df_non_semi_all.apply(assign_segment_2x2, axis=1)

print("\n✅ 비반도체 2×2 세그먼트 분류 완료")


[비반도체 세그먼트 분류] 2×2 매트릭스 준비

📊 분석 대상: 비반도체 74개사
시가총액 범위: 0.0조 ~ 709.8조

✅ 비반도체 2×2 세그먼트 분류 완료


In [ ]:
# [CELL 36] 2×2 매트릭스 요약 및 시각화
# ====================================================================

print("\n" + "="*100)
print("[2×2 세그먼트 매트릭스] 비반도체 업종")
print("="*100)

# 2×2 매트릭스 생성
matrix_2x2 = df_non_semi_all.groupby(['규모분류', '등급분류']).size().reset_index(name='기업수')
matrix_pivot = matrix_2x2.pivot(index='등급분류', columns='규모분류', values='기업수').fillna(0).astype(int)

# 행 순서 변경 (우수 먼저, 아래쪽에)
matrix_pivot = matrix_pivot.loc[['우수 (A/B+)', '부실 (C/D)']]

# 컬럼 순서 변경 (대형 먼저)
matrix_pivot = matrix_pivot[['대형 (1조↑)', '중소 (1조↓)']]

print("\n┌─────────────────┬──────────────────┐")
print(f"│ I-A 플래그십    │ I-B 대형개선사   │")
print(f"│ (대형+우수)     │ (대형+부실)      │")
print(f"│  {matrix_pivot.loc['우수 (A/B+)', '대형 (1조↑)']:.0f}개사       │  {matrix_pivot.loc['부실 (C/D)', '대형 (1조↑)']:.0f}개사        │")
print("├─────────────────┼──────────────────┤")
print(f"│ II-A 성장리더   │ II-B 표준화      │")
print(f"│ (중소+우수)     │ (중소+부실)      │")
print(f"│  {matrix_pivot.loc['우수 (A/B+)', '중소 (1조↓)']:.0f}개사       │  {matrix_pivot.loc['부실 (C/D)', '중소 (1조↓)']:.0f}개사        │")
print("└─────────────────┴──────────────────┘")

# 2x2 매트릭스 시각화 (히트맵)
print("\n" + "="*100)
print("[시각화] 2x2 비반도체 세그먼트 매트릭스 히트맵")
print("="*100)

fig_2x2_heatmap = go.Figure(data=go.Heatmap(
    z=matrix_pivot.values,
    x=matrix_pivot.columns,
    y=matrix_pivot.index,
    text=matrix_pivot.values,
    texttemplate='%{text}개사',
    textfont={"size": 14, "color": "white"},
    colorscale='RdYlGn',
    showscale=True,
    colorbar=dict(title="기업수", thickness=20, len=0.7)
))

fig_2x2_heatmap.update_layout(
    title_text='<b>[비반도체 세그먼트] 2×2 매트릭스 히트맵</b>',
    xaxis_title='규모 분류',
    yaxis_title='등급 분류',
    font=dict(family='Arial', size=12),
    height=500,
    width=700
)
fig_2x2_heatmap.show()

# 세부 통계
print("\n【세그먼트별 상세 현황】")
for segment in ['I-A (플래그십)', 'I-B (대형개선사)', 'II-A (성장리더)', 'II-B (표준화)']:
    subset = df_non_semi_all[df_non_semi_all['Segment'] == segment]
    if len(subset) > 0:
        avg_grade = subset['ESG등급'].value_counts().index[0]
        publish_rate = len(subset[subset['보고서_발행_유형'] != '미발행']) / len(subset) * 100
        print(f"\n【{segment}】{len(subset)}개사")
        print(f"  - 평균등급: {avg_grade}")
        print(f"  - 보고서발행율: {publish_rate:.1f}%")
        print(f"  - 평균시가총액: ₩{subset['시가총액_수치'].mean()/1e12:.1f}조")

print("\n✅ 2×2 비반도체 세그먼트 매트릭스 EDA 완료!")



[2×2 세그먼트 매트릭스] 비반도체 업종

┌─────────────────┬──────────────────┐
│ I-A 플래그십    │ I-B 대형개선사   │
│ (대형+우수)     │ (대형+부실)      │
│  20개사       │  11개사        │
├─────────────────┼──────────────────┤
│ II-A 성장리더   │ II-B 표준화      │
│ (중소+우수)     │ (중소+부실)      │
│  9개사       │  34개사        │
└─────────────────┴──────────────────┘

[시각화] 2x2 비반도체 세그먼트 매트릭스 히트맵



【세그먼트별 상세 현황】

【I-A (플래그십)】20개사
  - 평균등급: A
  - 보고서발행율: 95.0%
  - 평균시가총액: ₩48.3조

【I-B (대형개선사)】11개사
  - 평균등급: D
  - 보고서발행율: 45.5%
  - 평균시가총액: ₩3.8조

【II-A (성장리더)】9개사
  - 평균등급: B+
  - 보고서발행율: 66.7%
  - 평균시가총액: ₩0.6조

【II-B (표준화)】34개사
  - 평균등급: D
  - 보고서발행율: 32.4%
  - 평균시가총액: ₩0.3조

✅ 2×2 비반도체 세그먼트 매트릭스 EDA 완료!


In [ ]:
# [CELL 37] 2×2 매트릭스 히트맵 (ESG등급 분포)
# ====================================================================

print("\n" + "="*100)
print("[2×2 히트맵] 각 Segment의 ESG등급 분포")
print("="*100)

# 각 세그먼트별 등급 분포
heatmap_data = []
heatmap_labels = []

for seg in ['I-A (플래그십)', 'I-B (대형개선사)', 'II-A (성장리더)', 'II-B (표준화)']:
    subset = df_non_semi_all[df_non_semi_all['Segment'] == seg]
    if len(subset) > 0:
        grade_dist = subset['ESG등급'].value_counts().reindex(grade_order).fillna(0).astype(int)
        heatmap_data.append(grade_dist.values)
        heatmap_labels.append(seg)

fig_heatmap_2x2 = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=grade_order,
    y=heatmap_labels,
    text=heatmap_data,
    texttemplate='%{text}개사',
    textfont={"size": 14, "color": "white"},
    colorscale='RdYlGn',
    showscale=True,
    colorbar=dict(title="기업수")
))

fig_heatmap_2x2.update_layout(
    title='<b>[비반도체 2×2 매트릭스] 각 Segment의 ESG등급 분포</b>',
    xaxis_title='ESG 등급',
    yaxis_title='Segment',
    font=dict(family='Arial', size=12),
    height=500,
    width=900
)
fig_heatmap_2x2.show()

print("\n✅ 차트 생성: 2×2 히트맵")


[2×2 히트맵] 각 Segment의 ESG등급 분포



✅ 차트 생성: 2×2 히트맵


In [ ]:
# [CELL 38] 각 Segment의 기업 상세 목록
# ====================================================================

print("\n" + "="*100)
print("[각 Segment의 기업 상세 목록]")
print("="*100)

segment_order = ['I-A (플래그십)', 'I-B (대형개선사)', 'II-A (성장리더)', 'II-B (표준화)']
segment_info_2x2 = {
    'I-A (플래그십)': {'설명': '대형+우수', '전략': '💰 최우선 고객'},
    'I-B (대형개선사)': {'설명': '대형+부실', '전략': '📋 개선 기회'},
    'II-A (성장리더)': {'설명': '중소+우수', '전략': '🚀 성장 잠재'},
    'II-B (표준화)': {'설명': '중소+부실', '전략': '📱 저비용 모델'}
}

for segment in segment_order:
    subset = df_non_semi_all[df_non_semi_all['Segment'] == segment].sort_values('ESG등급')

    if len(subset) > 0:
        print(f"\n{'='*100}")
        print(f"【{segment}】{segment_info_2x2[segment]['설명']}")
        print(f"전략: {segment_info_2x2[segment]['전략']}")
        print(f"{'='*100}")
        print(f"총 {len(subset)}개사")
        print("-"*100)

        display_cols = ['기업명', 'ESG등급', '환경', '사회', '지배구조', '시가총액', '보고서_발행_유형']
        print(subset[display_cols].to_string(index=False))


[각 Segment의 기업 상세 목록]

【I-A (플래그십)】대형+우수
전략: 💰 최우선 고객
총 20개사
----------------------------------------------------------------------------------------------------
       기업명 ESG등급 환경 사회 지배구조         시가총액 보고서_발행_유형
  HD현대일렉트릭    A+ A+ A+    A 2.790050e+13    둘 다 발행
SK아이이테크놀로지     A  A A+   B+ 2.048770e+12    둘 다 발행
     LG이노텍     A  A A+    A 6.413790e+12    둘 다 발행
      대한전선     A  A  A   B+ 4.269640e+12    둘 다 발행
     두산퓨얼셀     A  A A+   B+ 1.889490e+12    둘 다 발행
  LG에너지솔루션     A B+ A+    A 8.622900e+13    둘 다 발행
      삼성전기     A  A A+   B+ 1.904690e+13    둘 다 발행
      삼성전자     A B+ A+   B+ 7.097646e+14    둘 다 발행
     삼성SDI     A  A A+   B+ 2.171780e+13    둘 다 발행
        심텍     A  A A+   B+ 1.842840e+12       미발행
      LG전자     A  A A+   B+ 1.496930e+13    둘 다 발행
   LG디스플레이     A  A A+   B+ 5.905000e+12    둘 다 발행
   엘에스일렉트릭     A A+  A    A 1.380000e+13    둘 다 발행
    포스코퓨처엠     A  A A+    A 1.663290e+13    둘 다 발행
     한화시스템     A  A A+   B+ 1.027720e+13    둘 다 발행
      가온전선    B+  A A

In [ ]:
# [CELL 39] 2×2 매트릭스 시각화 (파이 차트)
# ====================================================================

print("\n" + "="*100)
print("[2×2 대시보드] 각 Segment의 기업 분포")
print("="*100)

fig_dashboard_2x2 = make_subplots(
    rows=2, cols=2,
    subplot_titles=['I-A 플래그십\n(대형+우수)', 'I-B 대형개선사\n(대형+부실)',
                    'II-A 성장리더\n(중소+우수)', 'II-B 표준화\n(중소+부실)'],
    specs=[[{'type': 'pie'}, {'type': 'pie'}],
           [{'type': 'pie'}, {'type': 'pie'}]],
    vertical_spacing=0.2,
    horizontal_spacing=0.15
)

position = [(1, 1), (1, 2), (2, 1), (2, 2)]
colors = ['#1D3557', '#457B9D', '#A8DADC', '#F4A261', '#E76F51', '#E63946']

for segment, (row, col) in zip(segment_order, position):
    subset = df_non_semi_all[df_non_semi_all['Segment'] == segment]

    if len(subset) > 0:
        grade_dist = subset['ESG등급'].value_counts().reindex(grade_order).fillna(0).astype(int)
        grade_labels = [f"{g} ({grade_dist[g]})" for g in grade_order if grade_dist[g] > 0]
        grade_values = [grade_dist[g] for g in grade_order if grade_dist[g] > 0]

        grade_colors = []
        for g in grade_order:
            if grade_dist[g] > 0:
                grade_colors.append(colors[grade_order.index(g)])

        fig_dashboard_2x2.add_trace(
            go.Pie(
                labels=grade_labels,
                values=grade_values,
                marker=dict(colors=grade_colors),
                textposition='inside',
                textinfo='label+percent'
            ),
            row=row, col=col
        )

fig_dashboard_2x2.update_layout(
    title_text='<b>[비반도체 2×2 대시보드] 각 Segment의 ESG등급 분포</b>',
    height=800,
    width=1000,
    font=dict(family='Arial', size=10)
)
fig_dashboard_2x2.show()

print("\n✅ 차트 생성: 2×2 대시보드")


[2×2 대시보드] 각 Segment의 기업 분포



✅ 차트 생성: 2×2 대시보드


In [ ]:
# [CELL 40] 2×2 매트릭스 산업코드 분석
# ====================================================================

print("\n" + "="*100)
print("[산업코드 분석] 각 Segment의 주요 산업")
print("="*100)

for segment in segment_order:
    subset = df_non_semi_all[df_non_semi_all['Segment'] == segment]

    if len(subset) > 0:
        print(f"\n【{segment}】")
        industry_dist = subset['산업코드'].value_counts().head(5).reset_index()
        industry_dist.columns = ['산업코드', '기업수']

        # 산업명 매핑 (없으면 산업코드 사용)
        industry_dist['산업명'] = industry_dist['산업코드'].map(industry_code_map).fillna(industry_dist['산업코드'])
        industry_dist['산업표시'] = industry_dist.apply(lambda x: f"{x['산업명']} ({x['산업코드']})", axis=1)

        for idx, row in industry_dist.iterrows():
            print(f"  {idx+1}. {row['산업표시']}: {row['기업수']}개사")

        # 시각화 추가
        fig = px.bar(
            industry_dist,
            x='산업표시',
            y='기업수',
            title=f'<b>[{segment}] 주요 산업코드 분포 (상위 5개)</b>',
            labels={'기업수': '기업 수 (개사)', '산업표시': '산업명 (산업코드)'},
            color='기업수',
            color_continuous_scale='Viridis',
            height=500,
            width=800
        )
        fig.update_layout(
            xaxis_title=None, # X축 타이틀 제거
            font=dict(family='Arial', size=10),
            xaxis={'tickangle': -45},
            yaxis_range=[0, industry_dist['기업수'].max() * 1.2]
        )
        fig.show()
        print("\n" + "="*100)


[산업코드 분석] 각 Segment의 주요 산업

【I-A (플래그십)】
  1. 인쇄회로기판 및 전자부품 실장기판 제조업 (C2622): 3개사
  2. 운송장비용 이차전지 제조업 (C28202): 3개사
  3. 기타 가정용 전기기기 제조업 (C28302): 2개사
  4. C282 (C282): 2개사
  5. 통신 및 방송장비 제조업 (C264): 2개사




【I-B (대형개선사)】
  1. 기타 전자 부품 제조업 (C2629): 2개사
  2. 인쇄회로기판 및 전자부품 실장기판 제조업 (C2622): 2개사
  3. C289 (C289): 1개사
  4. C28519 (C28519): 1개사
  5. C28111 (C28111): 1개사




【II-A (성장리더)】
  1. 기타 전자 부품 제조업 (C2629): 3개사
  2. 그 외 기타 전자 부품 제조업 (C26299): 2개사
  3. C28520 (C28520): 1개사
  4. C26519 (C26519): 1개사
  5. 방송 및 무선 통신장비 제조업 (C2642): 1개사




【II-B (표준화)】
  1. 기타 전자 부품 제조업 (C2629): 5개사
  2. 통신 및 방송장비 제조업 (C264): 3개사
  3. 전자 축전기 제조업 (C26291): 3개사
  4. 인쇄회로기판 및 전자부품 실장기판 제조업 (C2622): 3개사
  5. 전자 부품 제조업 (C262): 3개사


In [ ]:
# [CELL 41] 벤치마킹 대상 추천 (2×2 방식)
# ====================================================================

print("\n" + "="*100)
print("[벤치마킹 대상 추천] 각 Segment별 1-2개사")
print("="*100)

for segment in segment_order:
    subset = df_non_semi_all[df_non_semi_all['Segment'] == segment]

    # 벤치마킹 조건: A등급 + 보고서 발행
    benchmark = subset[
        (subset['ESG등급'] == 'A') &
        (subset['보고서_발행_유형'].isin(['둘 다 발행', '지속가능경영']))
    ].sort_values('시가총액_수치', ascending=False).head(2)

    # 없으면 A+ 또는 B+
    if len(benchmark) == 0:
        benchmark = subset[
            (subset['ESG등급'].isin(['A+', 'B+'])) &
            (subset['보고서_발행_유형'] != '미발행')
        ].sort_values('시가총액_수치', ascending=False).head(2)

    print(f"\n【{segment}】")
    if len(benchmark) > 0:
        for idx, (_, company) in enumerate(benchmark.iterrows(), 1):
            print(f"  {idx}. {company['기업명']}")
            print(f"     ESG: {company['ESG등급']} | 시총: {company['시가총액']} | 보고서: {company['보고서_발행_유형']}")
    else:
        print(f"  ⚠️ 추천 기업 없음")


[벤치마킹 대상 추천] 각 Segment별 1-2개사

【I-A (플래그십)】
  1. 삼성전자
     ESG: A | 시총: 709764586847800.0 | 보고서: 둘 다 발행
  2. LG에너지솔루션
     ESG: A | 시총: 86229000000000.0 | 보고서: 둘 다 발행

【I-B (대형개선사)】
  ⚠️ 추천 기업 없음

【II-A (성장리더)】
  1. 해성디에스
     ESG: A | 시총: 946900000000.0 | 보고서: 둘 다 발행

【II-B (표준화)】
  ⚠️ 추천 기업 없음


In [ ]:
# [CELL 42] 컨설팅 후보 추천 (2×2 방식)
# ====================================================================

print("\n" + "="*100)
print("[컨설팅 후보 추천] 각 Segment별 3-5개사")
print("="*100)

for segment in segment_order:
    subset = df_non_semi_all[df_non_semi_all['Segment'] == segment]

    if len(subset) > 0:
        # 컨설팅 후보: 같은 세그먼트 내에서 우선순위
        consulting = subset.sort_values('ESG등급').head(5)

        print(f"\n【{segment}】({len(subset)}개사 중 상위 5개)")
        print("-"*100)
        display_cols = ['기업명', 'ESG등급', '환경', '지배구조', '보고서_발행_유형']
        print(consulting[display_cols].to_string(index=False))



[컨설팅 후보 추천] 각 Segment별 3-5개사

【I-A (플래그십)】(20개사 중 상위 5개)
----------------------------------------------------------------------------------------------------
       기업명 ESG등급 환경 지배구조 보고서_발행_유형
  HD현대일렉트릭    A+ A+    A    둘 다 발행
SK아이이테크놀로지     A  A   B+    둘 다 발행
     LG이노텍     A  A    A    둘 다 발행
      대한전선     A  A   B+    둘 다 발행
     두산퓨얼셀     A  A   B+    둘 다 발행

【I-B (대형개선사)】(11개사 중 상위 5개)
----------------------------------------------------------------------------------------------------
   기업명 ESG등급 환경 지배구조 보고서_발행_유형
이수페타시스     B B+    D    둘 다 발행
 효성중공업     B  A    D    둘 다 발행
 서진시스템     C  C    D       미발행
에스앤에스텍     C  D   B+       미발행
코리아써키트     C  B    D    둘 다 발행

【II-A (성장리더)】(9개사 중 상위 5개)
----------------------------------------------------------------------------------------------------
    기업명 ESG등급 환경 지배구조 보고서_발행_유형
 한솔테크닉스     A B+    A     지배구조만
  해성디에스     A  A   B+    둘 다 발행
  경동나비엔    B+  A    B    둘 다 발행
 대주전자재료    B+ B+    B   지속가능경영만
솔루스첨단소재    B+ B+    B     

In [ ]:
# [CELL 43] 2×2 세그먼트 특성 요약
# ====================================================================

print("\n" + "="*100)
print("[2×2 세그먼트 특성 요약]")
print("="*100)

summary_2x2 = []

for segment in segment_order:
    subset = df_non_semi_all[df_non_semi_all['Segment'] == segment]

    if len(subset) > 0:
        row = {
            'Segment': segment,
            '기업수': len(subset),
            'A등급': len(subset[subset['ESG등급'] == 'A']),
            'B등급': len(subset[subset['ESG등급'].isin(['B+', 'B'])]),
            'C/D등급': len(subset[subset['ESG등급'].isin(['C', 'D'])]),
            '보고서발행': len(subset[subset['보고서_발행_유형'] != '미발행']),
            '평균시총(조)': round(subset['시가총액_수치'].mean() / 1e12, 1)
        }
        summary_2x2.append(row)

summary_df_2x2 = pd.DataFrame(summary_2x2)

print("\n【비반도체 2×2 세그먼트 요약】")
print(summary_df_2x2.to_string(index=False))



[2×2 세그먼트 특성 요약]

【비반도체 2×2 세그먼트 요약】
    Segment  기업수  A등급  B등급  C/D등급  보고서발행  평균시총(조)
 I-A (플래그십)   20   14    5      0     19     48.3
I-B (대형개선사)   11    0    2      9      5      3.8
II-A (성장리더)    9    2    7      0      6      0.6
 II-B (표준화)   34    0    4     30     11      0.3


In [ ]:
# [CELL 44] 2×2 전략 매핑
# ====================================================================

print("\n" + "="*100)
print("[2×2 세그먼트 전략]")
print("="*100)

strategies_2x2 = {
    'I-A (플래그십)': {
        '설명': '대형+우수기업',
        '특성': '높은 ESG등급, 보고서 발행, 자금 여력 충분',
        '전략': '💰 딥다이브 고비용 컨설팅',
        '세일즈': '공급망 ESG, 글로벌 평가 개선',
        '우선순위': '⭐⭐⭐⭐⭐ (최우선)'
    },
    'I-B (대형개선사)': {
        '설명': '대형+부실기업',
        '특성': '낮은 ESG등급, 보고서 미발행, 개선 동기 필요',
        '전략': '📋 개선 기회 강조형',
        '세일즈': '등급 하락 리스크, 투자자 이탈 방지',
        '우선순위': '⭐⭐⭐⭐ (높음)'
    },
    'II-A (성장리더)': {
        '설명': '중소+우수기업',
        '특성': '높은 ESG등급, 성장 가능성',
        '전략': '🚀 성장 지원형',
        '세일즈': '공급망 네트워크, 경쟁사 벤치마킹',
        '우선순위': '⭐⭐⭐ (중간)'
    },
    'II-B (표준화)': {
        '설명': '중소+부실기업',
        '특성': '낮은 ESG등급, 낮은 투자 여력',
        '전략': '📱 저비용 확장형',
        '세일즈': '자가진단 도구, 온라인 교육',
        '우선순위': '⭐⭐ (선별)'
    }
}

for segment, info in strategies_2x2.items():
    print(f"\n【{segment}】")
    print(f"  특성: {info['특성']}")
    print(f"  전략: {info['전략']}")
    print(f"  세일즈: {info['세일즈']}")
    print(f"  우선순위: {info['우선순위']}")


[2×2 세그먼트 전략]

【I-A (플래그십)】
  특성: 높은 ESG등급, 보고서 발행, 자금 여력 충분
  전략: 💰 딥다이브 고비용 컨설팅
  세일즈: 공급망 ESG, 글로벌 평가 개선
  우선순위: ⭐⭐⭐⭐⭐ (최우선)

【I-B (대형개선사)】
  특성: 낮은 ESG등급, 보고서 미발행, 개선 동기 필요
  전략: 📋 개선 기회 강조형
  세일즈: 등급 하락 리스크, 투자자 이탈 방지
  우선순위: ⭐⭐⭐⭐ (높음)

【II-A (성장리더)】
  특성: 높은 ESG등급, 성장 가능성
  전략: 🚀 성장 지원형
  세일즈: 공급망 네트워크, 경쟁사 벤치마킹
  우선순위: ⭐⭐⭐ (중간)

【II-B (표준화)】
  특성: 낮은 ESG등급, 낮은 투자 여력
  전략: 📱 저비용 확장형
  세일즈: 자가진단 도구, 온라인 교육
  우선순위: ⭐⭐ (선별)


In [ ]:
# [CELL 45] 최종 액션 플랜
# ====================================================================

print("\n" + "="*100)
print("[최종 요약] 비반도체 2×2 세그먼트 분류 완료! 🎉")
print("="*100)

print("""
✅ 비반도체 업종 2×2 분류 완료!

【분류 기준】
┌─────────────────┬──────────────────┐
│  규모 기준      │  등급 기준       │
├─────────────────┼──────────────────┤
│ 1조↑: 대형      │ A/B+: 우수       │
│ 1조↓: 중소      │ C/D: 부실        │
└─────────────────┴──────────────────┘

【결과 Segment】

I-A 플래그십 (대형+우수)
  └─ 💰 고비용 딥다이브
  └─ 공급망 ESG
  └─ 우선순위: ⭐⭐⭐⭐⭐

I-B 대형개선사 (대형+부실)
  └─ 📋 개선 기회 강조
  └─ 리스크 관리
  └─ 우선순위: ⭐⭐⭐⭐

II-A 성장리더 (중소+우수)
  └─ 🚀 성장 지원
  └─ 벤치마킹
  └─ 우선순위: ⭐⭐⭐

II-B 표준화 (중소+부실)
  └─ 📱 저비용 확장
  └─ 자가진단 도구
  └─ 우선순위: ⭐⭐

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📌 기존 ABCD vs 신규 2×2 비교

기존 분류 (전체 89개사):
  A (플래그십, 18개) / B (성장, 16개) / C (기술, 27개) / D (표준, 28개)
  → 시가총액 + 업종 기준

신규 분류 (비반도체 76개):
  I-A / I-B / II-A / II-B
  → 시가총액 + ESG등급 기준
  → 더 세분화, 비반도체에 특화

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎯 다음 단계:

1️⃣ 각 2×2 Segment별 벤치마킹 대상 확정
2️⃣ 컨설팅 후보 최종 선정
3️⃣ 개별 기업 프로필 작성
4️⃣ 마케팅 메시지 개발

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("="*100)
print("✅ 비반도체 2×2 세그먼트 분석 완료!")
print("="*100)


[최종 요약] 비반도체 2×2 세그먼트 분류 완료! 🎉

✅ 비반도체 업종 2×2 분류 완료!
 
【분류 기준】
┌─────────────────┬──────────────────┐
│  규모 기준      │  등급 기준       │
├─────────────────┼──────────────────┤
│ 1조↑: 대형      │ A/B+: 우수       │
│ 1조↓: 중소      │ C/D: 부실        │
└─────────────────┴──────────────────┘
 
【결과 Segment】
 
I-A 플래그십 (대형+우수)
  └─ 💰 고비용 딥다이브
  └─ 공급망 ESG
  └─ 우선순위: ⭐⭐⭐⭐⭐
 
I-B 대형개선사 (대형+부실)
  └─ 📋 개선 기회 강조
  └─ 리스크 관리
  └─ 우선순위: ⭐⭐⭐⭐
 
II-A 성장리더 (중소+우수)
  └─ 🚀 성장 지원
  └─ 벤치마킹
  └─ 우선순위: ⭐⭐⭐
 
II-B 표준화 (중소+부실)
  └─ 📱 저비용 확장
  └─ 자가진단 도구
  └─ 우선순위: ⭐⭐
 
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 
📌 기존 ABCD vs 신규 2×2 비교
 
기존 분류 (전체 89개사):
  A (플래그십, 18개) / B (성장, 16개) / C (기술, 27개) / D (표준, 28개)
  → 시가총액 + 업종 기준
 
신규 분류 (비반도체 76개):
  I-A / I-B / II-A / II-B
  → 시가총액 + ESG등급 기준
  → 더 세분화, 비반도체에 특화
 
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 
🎯 다음 단계:
 
1️⃣ 각 2×2 Segment별 벤치마킹 대상 확정
2️⃣ 컨설팅 후보 최종 선정
3️⃣ 개별 기업 프로필 작성
4️⃣ 마케팅 메시지 개발
 
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# [CELL 32] Segment별 대시보드 (4개 파이 차트)
# ====================================================================

print("\n" + "="*100)
print("[Segment별 대시보드] 각 Segment의 ESG 등급 분포")
print("="*100)

segment_info = {
    'A': ('플래그십', '대형 (3조 이상)'),
    'B': ('성장 가능', '중형 (1~3조)'),
    'C': ('기술 특화', '소형 (3천억~1조)'),
    'D': ('표준화', '초소형 (3천억 미만)')
}

fig_segment_dashboard = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"{name} ({market_cap})" for name, market_cap in segment_info.values()],
    specs=[[{'type': 'pie'}, {'type': 'pie'}],
           [{'type': 'pie'}, {'type': 'pie'}]],
    vertical_spacing=0.15,
    horizontal_spacing=0.15
)

position = [(1, 1), (1, 2), (2, 1), (2, 2)]

for (seg_code, seg_name), (row, col) in zip(segment_info.items(), position):
    market_cap = segment_info[seg_code][1]
    subset = df_segment_filtered[df_segment_filtered['시가총액분류'] == market_cap]

    grade_dist = subset['ESG등급'].value_counts().reindex(grade_order).fillna(0).astype(int)
    grade_labels = [f"{g} ({grade_dist[g]})" for g in grade_order if grade_dist[g] > 0]
    grade_values = [grade_dist[g] for g in grade_order if grade_dist[g] > 0]

    colors = ['#1D3557', '#457B9D', '#A8DADC', '#F4A261', '#E76F51', '#E63946']
    grade_colors = []
    for g in grade_order:
        if grade_dist[g] > 0:
            grade_colors.append(colors[grade_order.index(g)])

    fig_segment_dashboard.add_trace(
        go.Pie(
            labels=grade_labels,
            values=grade_values,
            name=f"Segment {seg_code}",
            marker=dict(colors=grade_colors),
            textposition='inside',
            textinfo='label+percent'
        ),
        row=row, col=col
    )

fig_segment_dashboard.update_layout(
    title_text='<b>[Segment 대시보드] 각 Segment의 ESG 등급 분포</b>',
    height=800,
    width=1000,
    font=dict(family='Arial', size=11)
)
fig_segment_dashboard.show()

print("\n✅ 차트 생성: Segment 대시보드")


[Segment별 대시보드] 각 Segment의 ESG 등급 분포



✅ 차트 생성: Segment 대시보드


In [ ]:
# [CELL 32] Segment별 대시보드 (4개 파이 차트)
# ====================================================================

print("\n" + "="*100)
print("[Segment별 대시보드] 각 Segment의 ESG 등급 분포")
print("="*100)

segment_info = {
    'A': ('플래그십', '대형 (3조 이상)'),
    'B': ('성장 가능', '중형 (1~3조)'),
    'C': ('기술 특화', '소형 (3천억~1조)'),
    'D': ('표준화', '초소형 (3천억 미만)')
}

fig_segment_dashboard = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"{name} ({market_cap})" for name, market_cap in segment_info.values()],
    specs=[[{'type': 'pie'}, {'type': 'pie'}],
           [{'type': 'pie'}, {'type': 'pie'}]],
    vertical_spacing=0.15,
    horizontal_spacing=0.15
)

position = [(1, 1), (1, 2), (2, 1), (2, 2)]

for (seg_code, seg_name), (row, col) in zip(segment_info.items(), position):
    market_cap = segment_info[seg_code][1]
    subset = df_segment_filtered[df_segment_filtered['시가총액분류'] == market_cap]

    grade_dist = subset['ESG등급'].value_counts().reindex(grade_order).fillna(0).astype(int)
    grade_labels = [f"{g} ({grade_dist[g]})" for g in grade_order if grade_dist[g] > 0]
    grade_values = [grade_dist[g] for g in grade_order if grade_dist[g] > 0]

    colors = ['#1D3557', '#457B9D', '#A8DADC', '#F4A261', '#E76F51', '#E63946']
    grade_colors = []
    for g in grade_order:
        if grade_dist[g] > 0:
            grade_colors.append(colors[grade_order.index(g)])

    fig_segment_dashboard.add_trace(
        go.Pie(
            labels=grade_labels,
            values=grade_values,
            name=f"Segment {seg_code}",
            marker=dict(colors=grade_colors),
            textposition='inside',
            textinfo='label+percent'
        ),
        row=row, col=col
    )

fig_segment_dashboard.update_layout(
    title_text='<b>[Segment 대시보드] 각 Segment의 ESG 등급 분포</b>',
    height=800,
    width=1000,
    font=dict(family='Arial', size=11)
)
fig_segment_dashboard.show()

print("\n✅ 차트 생성: Segment 대시보드")


[Segment별 대시보드] 각 Segment의 ESG 등급 분포



✅ 차트 생성: Segment 대시보드


# **[STEP 3] 세부 후보군 정하기**
- Segment별 우선순위 정렬
- 기업별 특성 분석
- 최종 타겟팅 전략 수립